In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity




In [2]:
restaurants=pd.read_csv(r"C:\Users\nagpa\OneDrive\Desktop\Cognizant projects\Dataset.csv")
print(restaurants.head())

   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

In [3]:
print(restaurants.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [4]:
print(restaurants.describe())

       Restaurant ID  Country Code    Longitude     Latitude  \
count   9.551000e+03   9551.000000  9551.000000  9551.000000   
mean    9.051128e+06     18.365616    64.126574    25.854381   
std     8.791521e+06     56.750546    41.467058    11.007935   
min     5.300000e+01      1.000000  -157.948486   -41.330428   
25%     3.019625e+05      1.000000    77.081343    28.478713   
50%     6.004089e+06      1.000000    77.191964    28.570469   
75%     1.835229e+07      1.000000    77.282006    28.642758   
max     1.850065e+07    216.000000   174.832089    55.976980   

       Average Cost for two  Price range  Aggregate rating         Votes  
count           9551.000000  9551.000000       9551.000000   9551.000000  
mean            1199.210763     1.804837          2.666370    156.909748  
std            16121.183073     0.905609          1.516378    430.169145  
min                0.000000     1.000000          0.000000      0.000000  
25%              250.000000     1.000000        

In [5]:
#handle the missing values
restaurants['Cuisines']=restaurants['Cuisines'].fillna("Unknown")
restaurants['Price range']=restaurants['Price range'].fillna(restaurants['Price range'].mode()[0])
restaurants['Aggregate rating']=restaurants['Aggregate rating'].fillna(restaurants['Aggregate rating'].mean())
print(restaurants.head())

   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

In [6]:
#encode catagorical varaibles
label_encoder=LabelEncoder()
restaurants['cuisine_encoded']=label_encoder.fit_transform(restaurants['Cuisines'])
restaurants['Price_encoded']=label_encoder.fit_transform(restaurants['Price range'])
print(restaurants[['Cuisines', 'cuisine_encoded']].head())
print(restaurants[['Price range','Price_encoded']].head())

                           Cuisines  cuisine_encoded
0        French, Japanese, Desserts              920
1                          Japanese             1111
2  Seafood, Asian, Filipino, Indian             1671
3                   Japanese, Sushi             1126
4                  Japanese, Korean             1122
   Price range  Price_encoded
0            3              2
1            3              2
2            4              3
3            4              3
4            4              3


In [7]:
#create combined features
restaurants['Features']=(
    restaurants['Cuisines'].astype(str)+' '+
    restaurants['Price range'].astype(str)+' '+
    restaurants['Aggregate rating'].astype(str))
print(restaurants[['Restaurant Name', 'Features']].head())

          Restaurant Name                                Features
0        Le Petit Souffle        French, Japanese, Desserts 3 4.8
1        Izakaya Kikufuji                          Japanese 3 4.5
2  Heat - Edsa Shangri-La  Seafood, Asian, Filipino, Indian 4 4.4
3                    Ooma                   Japanese, Sushi 4 4.9
4             Sambo Kojin                  Japanese, Korean 4 4.8


In [8]:
#convert text to vectors
cv=CountVectorizer()
feature_matrix=cv.fit_transform(restaurants['Features'])
print(feature_matrix.shape)

(9551, 151)


In [9]:
#calculate cosine similarity
similarity=cosine_similarity(feature_matrix)
print(similarity)

[[1.         0.57735027 0.         ... 0.         0.         0.        ]
 [0.57735027 1.         0.         ... 0.         0.         0.        ]
 [0.         0.         1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.70710678]
 [0.         0.         0.         ... 0.         0.70710678 1.        ]]


In [15]:
#reccomndation function
def recommend_restaurant(Cuisines, price_range):
    user_input=Cuisines+' '+price_range
    user_vector=cv.transform([user_input])
    similarity_scores=cosine_similarity(user_vector,feature_matrix)
    recommended_scores=similarity_scores.argsort()[0][-10:][::-1]
    recommendations=restaurants.iloc[recommended_scores][[
        'Restaurant Name',
        'Cuisines',
        'Price range',
        'Aggregate rating']]
    return recommendations
result=recommend_restaurant('European,Asian,Indian','4')
print(result)

                               Restaurant Name  \
8     Spiral - Sofitel Philippine Plaza Manila   
7541        Capital Kitchen - Taj Palace Hotel   
1443          Ninkasi Imperial Brews & Cookery   
772                          Michael's Kitchen   
598           Grand Barbeque Buffet Restaurant   
6845                               Dirty Apron   
6                                   Buffet 101   
1852                Sandys Cocktails & Kitchen   
8047                United Coffee House Rewind   
3116                       United Coffee House   

                                          Cuisines  Price range  \
8                          European, Asian, Indian            4   
7541                 North Indian, European, Asian            4   
1443                 North Indian, Asian, European            4   
772                  North Indian, Asian, European            2   
598                                  Indian, Asian            3   
6845                               European, As